In [ ]:
from datetime import timedelta

from snowflake.snowpark import Session

from snowflake.snowpark.context import get_active_session
import json

In [ ]:
# session  = Session.builder.config("connection_name", "default").create()

In [ ]:
folder = r'workflows/'

Using SP to call mappings notebook


In [ ]:
session = get_active_session()

In [ ]:
session.sql("""
    CREATE OR REPLACE PROCEDURE EXECUTE_NOTEBOOK(NOTEBOOK_NAME STRING)
    RETURNS STRING
    LANGUAGE SQL
    AS
    $$
    BEGIN
        EXECUTE IMMEDIATE 'EXECUTE NOTEBOOK ' || NOTEBOOK_NAME || '();';
        RETURN 'Notebook ' || NOTEBOOK_NAME || ' executed successfully!';
    END;
    $$;
""").collect()

Using SP to call the subworkflow task

In [ ]:
session.sql("""
    CREATE OR REPLACE PROCEDURE Call_Sub_Task_Graph(TASK_NAME STRING)
    RETURNS STRING
    LANGUAGE SQL
    AS
    $$
    BEGIN
        EXECUTE IMMEDIATE 'EXECUTE TASK ' || TASK_NAME ||';';
        RETURN 'TASK ' || TASK_NAME || ' executed successfully!';
    END;
    $$;
    """).collect()


In [ ]:
# database = session.get_current_database()
# schema = session.get_current_schema()
# warehouse = session.get_current_warehouse()
# folder = "workflows"
database = "Test"
schema = "PUBLIC"
warehouse = session.get_current_warehouse()
folder = "workflows"

In [ ]:
from contextlib import contextmanager
from snowflake.snowpark.exceptions import SnowparkSQLException

@contextmanager
def snowflake_exception_handler():
    try:
        yield
    except SnowparkSQLException as e:
        print(f"Error executing Snowflake query: {e}")

In [ ]:
class TaskGraph:
    database = "TEST"
    schema = "PUBLIC"
    warehouse = "COMPUTE_WH"
    folder = "workflows"
    def __init__(self, folder):
        self.folder = folder
        self.counter = 0
        self.last_parent = None

    def start_create(self, main_workflow_file):
        self.file_opner(main_workflow_file)

    def parser(self, file_data:dict, depends_on):
        last_child = list(file_data)[-1]
        for task_name in file_data:
            self.create_task(task_name, file_data, depends_on=depends_on)
            depends_on=None
        if file_data[last_child]["stepType"] == "session":
            self.last_parent = last_child

    def file_opner(self, file_name:str, parent_dependency=None):
        folder = self.folder + "/" if self.folder!="" else self.folder
        with open(folder + file_name + ".json") as f:
            file_data = json.load(f)
            # first_child = next(iter(file_data))
            last_child = self.parser(file_data, depends_on=parent_dependency)
        

    def create_task(self, task_name, data, depends_on=None):
        self.counter+=1
        if depends_on==None:
            depends_on = data[task_name]['depends_on']
        notebook = data[task_name]['notebook']
        if  '..../' in notebook:
            notebook = notebook.removeprefix('..../')
        print(f"Task {self.counter} : {task_name} depends on {depends_on}")

        
        if data[task_name]["stepType"] == "Sub Workflow":
            with snowflake_exception_handler():
                print(f"\nEntering SUB WORKFLOW: {task_name} and depends on {depends_on}")
                self.file_opner(file_name=task_name, parent_dependency=task_name)
                

                    
                print(f"Created SUB WORKFLOW: {task_name} which depends on {depends_on}")
        
        elif data[task_name]["stepType"] == "session":
            print(depends_on)
            if self.last_parent!=None:
                depends_on=self.last_parent
                self.last_parent=None
            with snowflake_exception_handler():
                print(f"CREATING session : {task_name} which depends on {depends_on}")
                session.sql(
                    f"""
                        CREATE OR REPLACE TASK {database}.{schema}.{task_name}
                        WAREHOUSE = {warehouse}
                        AFTER {database}.{schema}.{depends_on}
                        AS
                        CALL EXECUTE_NOTEBOOK('{notebook}');
                    """).collect()
                # self.last_parent=None
                print(task_name, " notebook task created successfully")

        elif data[task_name]["stepType"] == "mainWF":
            with snowflake_exception_handler():
                print(f"DATABASE: {database}, SCHEMA: {schema}, WAREHOUSE:{warehouse}")
                print(f"CREATING Main WORKFLOW: {task_name} which depends on {depends_on}")
                session.sql(
                    f"""create or replace task {database}.{schema}.{task_name}
                    warehouse={warehouse}
                    schedule='999 minute'
                    as SELECT 1;
                    """).collect()
                self.last_parent = task_name
                print(task_name, " root task is successfully created")
            
        

In [ ]:
a = TaskGraph("workflow_jsons")
a.start_create("mainworkflow")